# Exploración de técnicas de prompt-engineering para generar contra-narrativas

Notebook principal. Orquesta el pipeline definido en el paquete `cn_pipeline`:

1. Carga la configuración desde `config.example.json` (o un JSON propio).
2. Carga el dataset ASOHMO y el modelo de lenguaje.
3. Genera contra-narrativas con las técnicas de prompt seleccionadas.
4. Evalúa con BLEU/ROUGE/BERTScore y con el evaluador LLM-as-judge (obligatorio).
5. Guarda resultados en `outputs/<experimento>/`.

Se corre **un modelo por ejecución** (para evaluar otro modelo, cambiar `model_name` en el config o pasarlo por CLI con `--model_name`).

> Requiere GPU (ejecutar en CCAD/Colab). En CPU también corre, pero es muy lento.

In [ ]:
import os
import sys

# Asegura que la raíz del repo esté en sys.path (por si la notebook se mueve).
REPO_ROOT = os.path.abspath(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from cn_pipeline.config import load_config, config_to_dict
from cn_pipeline.pipeline import run_experiment

## 0. Credenciales del evaluador (LLM-as-judge)

El evaluador es obligatorio y consulta el endpoint del CCAD. El token se lee de la variable de entorno `LLM_JUDGE_API_KEY`: podés setearlo antes (`export LLM_JUDGE_API_KEY=...`) o dejar que la celda lo pida. El token **no** se guarda en la notebook.

In [ ]:
import getpass

if not os.environ.get("LLM_JUDGE_API_KEY"):
    os.environ["LLM_JUDGE_API_KEY"] = getpass.getpass("Token LLM-judge (CCAD): ")

## 0.1 Conexión con HuggingFace (opcional, para modelos gated)

Solo hace falta si el modelo es *gated* (p. ej. Llama; Gemma 4 es Apache-2.0). El token se lee de `HF_TOKEN` y, si no está seteado, la celda lo pide. El token **no** se guarda en la notebook. Recordá aceptar la licencia del modelo en la web con la misma cuenta.

In [ ]:
import getpass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Token HuggingFace (hf_...): ")

## 1. Configuración

Editá `config.example.json` (o copialo a `config.json`) y ajustá modelo, técnicas de prompt, nº de ejemplos few-shot, métricas, seed, etc.

In [ ]:
import json

CONFIG_PATH = "config.example.json"
cfg = load_config(CONFIG_PATH)
print(json.dumps(config_to_dict(cfg), ensure_ascii=False, indent=2))

## 2. Ejecutar el experimento

In [ ]:
detailed, summary, out_dir = run_experiment(cfg)

## 3. Resultados

Los archivos `results.csv`, `summary.csv`, `config.json` y `anecdotal.txt` quedan en `outputs/<experimento>/`. Para comparar técnicas (y más adelante modelos): `python compare_experiments.py`.

In [ ]:
summary